In [2]:
import pandas as pd
import joblib

X_train = pd.read_parquet("../data/processed/X_train.parquet")
X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_train = pd.read_parquet("../data/processed/y_train.parquet")['Churn']
y_test = pd.read_parquet("../data/processed/y_test.parquet")['Churn']
preprocessor = joblib.load("../data/processed/preprocessor.joblib")

## Model Comparison - LogisticRegression vs RandomForest Classifier

Baseline comparison of both models ('class_weight='balanced', from previous decision) using 5-folds `StratifiedKFold` cross-validation on the training set. F1 and recall (minority class) are tracked

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

logreg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000))
])

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42))
])

In [4]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg_scores = cross_validate(logreg_pipeline, X_train, y_train, cv=cv, scoring=['accuracy', 'precision', 'recall', 'f1'], return_train_score=True)

rf_scores = cross_validate(rf_pipeline, X_train, y_train, cv=cv, scoring=['accuracy', 'precision', 'recall', 'f1'], return_train_score=True)

In [5]:
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [logreg_scores['test_accuracy'].mean(), rf_scores['test_accuracy'].mean()],
    'Precision': [logreg_scores['test_precision'].mean(), rf_scores['test_precision'].mean()],
    'Recall': [logreg_scores['test_recall'].mean(), rf_scores['test_recall'].mean()],
    'F1-Score': [logreg_scores['test_f1'].mean(), rf_scores['test_f1'].mean()]
})

print(comparison_df)

                 Model  Accuracy  Precision    Recall  F1-Score
0  Logistic Regression  0.749914   0.518611  0.803344  0.630239
1        Random Forest  0.790916   0.646413  0.469565  0.543921


**Findings:** LogisticRegression catches 80% of churners; RandomForest catches 47%. That's the number that decides this, a missed churner causes no chance of intervention, so recall matters more so than accuracy or precision RandomForest wins on (79.1% vs 75.0% accuracy, 0.646 vs 0.519 precision). F1 actually favors RandomForest (0.544 vs 0.630 for LogReg), but F1 is the wrong lens here as it rewards a precision/recall balance this problem doesn't need.

The gap probably comes down to how `class_weight='balanced'` hits each model differently. LogisticRegression reweights the loss directly, so it's a strong, blunt push toward flagging minority-class rows. RandomForest reweights impurity at each split instead, and that effect thins out once you're averaging votes across 100 trees. Neither model is tuned yet, so this isn't the two algorithms at their best but at default setting up

**Decision:** LogisticRegression is the model going forward. RandomForest stays on the
bench. If tuning LogReg stalls, or if false positives start costing more than assumed,
it's the first thing to revisit.

## Hyperparameter Tuning

Tuned LogisticRegression's `C`, `penalty`, and `solver` with `GridSearchCV`, same 5-fold `StratifiedKFold` as Step 6, scoring on recall per the Step 4 decision. `lbfgs` only supports `l2`, so the grid split into two subspaces (`lbfgs`+`l2`, and `liblinear`/`saga`+`l1`/`l2`). 25 valid combos total — small enough to search exhaustively instead of sampling.

In [8]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_distributions = [
    {
        "classifier__solver": ["lbfgs"],
        "classifier__penalty": ["l2"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
    {
        "classifier__solver": ["liblinear", "saga"],
        "classifier__penalty": ["l1", "l2"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
]

random_search = RandomizedSearchCV(
    logreg_pipeline, 
    param_distributions, 
    n_iter=10, 
    cv=cv, 
    scoring='f1', 
    random_state=42
)

random_search.fit(X_train, y_train)

print(random_search.best_params_)
print(random_search.best_score_)

tuned_scores = cross_validate(
    random_search.best_estimator_, 
    X_train, 
    y_train, 
    cv=cv, 
    scoring=['accuracy', 'precision', 'recall', 'f1'], 
    return_train_score=True
)

{'classifier__solver': 'lbfgs', 'classifier__penalty': 'l2', 'classifier__C': 0.01}
0.6401154026448651


In [9]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    logreg_pipeline,
    param_grid=param_distributions,
    cv=cv,
    scoring="recall",
    return_train_score=True,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

{'classifier__C': 0.1, 'classifier__penalty': 'l1', 'classifier__solver': 'liblinear'}
0.8127090301003344


In [13]:
tuned_scores = cross_validate(
    grid_search.best_estimator_, 
    X_train, 
    y_train, 
    cv=cv, 
    scoring=['accuracy', 'precision', 'recall', 'f1'], 
    return_train_score=True
)

print({k: v.mean() for k, v in tuned_scores.items() if 'test_' in k})

grid_search.best_estimator_.named_steps['classifier'].coef_

{'test_accuracy': np.float64(0.7518673729434626), 'test_precision': np.float64(0.5210822748457613), 'test_recall': np.float64(0.8127090301003344), 'test_f1': np.float64(0.6349150415824765)}


array([[-0.50161645,  0.38432477, -0.02702586,  0.37422726,  0.        ,
         0.        ,  0.1351091 ,  0.        , -0.14982492, -0.55751671,
         0.        ,  0.21636295,  0.59368981, -0.0053374 , -0.00390309,
        -0.36789594, -0.19067597, -0.11930238, -0.02422549,  0.        ,
        -0.01786845, -0.29212626, -0.26639542,  0.14145503, -0.01979364,
         0.15621789, -0.6428972 , -1.42820845,  0.323854  ,  0.        ,
         0.38053456,  0.        ,  0.        , -0.20852431,  0.        ,
         0.        ,  0.        ]])

Best params: `C=0.1`, `penalty='l1'`, `solver='liblinear'`.

| Metric | Step 6 baseline | Tuned |
|---|---|---|
| Accuracy | 0.750 | 0.752 |
| Precision | 0.519 | 0.521 |
| Recall | 0.803 | 0.813 |
| F1 | 0.630 | 0.635 |

All four metrics improved, not just recall. Train and test scores are close on every metric (recall 0.813 both), so the regularization isn't over- or under-fitting.

In [14]:
feature_names = grid_search.best_estimator_.named_steps['preprocessor'].get_feature_names_out()
coefs = grid_search.best_estimator_.named_steps['classifier'].coef_[0]
pd.Series(coefs, index=feature_names).sort_values()

cat__Contract_Two year                       -1.428208
cat__Contract_One year                       -0.642897
cat__PhoneService_Yes                        -0.557517
num__tenure                                  -0.501616
cat__OnlineSecurity_Yes                      -0.367896
cat__TechSupport_Yes                         -0.292126
cat__StreamingTV_No internet service         -0.266395
cat__tenure_bucket_24-35                     -0.208524
cat__OnlineBackup_No internet service        -0.190676
cat__Dependents_Yes                          -0.149825
cat__OnlineBackup_Yes                        -0.119302
num__TotalCharges                            -0.027026
cat__DeviceProtection_No internet service    -0.024225
cat__StreamingMovies_No internet service     -0.019794
cat__TechSupport_No internet service         -0.017868
cat__InternetService_No                      -0.005337
cat__OnlineSecurity_No internet service      -0.003903
cat__MultipleLines_No phone service           0.000000
cat__tenur

`l1` zeroed 11 of 37 coefficients, including two of six `..._No internet service` dummies (`DeviceProtection`, `StreamingMovies`). Those six columns duplicate each other, all flip to 1 together when a customer has no internet service, so `l1` collapsed real redundancy, not just weak signal.

Largest coefficients: `Contract_Two year` (-1.43), `Contract_One year` (-0.64), `tenure` (-0.50), `InternetService_Fiber optic` (+0.59). Contract length dominates the model.

**Decision:** tuned LogisticRegression (`C=0.1`, `l1`, `liblinear`) replaces the Step 6 baseline. RandomForest stays the untuned fallback.